# Retail Sales SQL — Solutions, Results & Business Findings

This notebook turns the 13 questions in `sql_query_p1.sql` into a repeatable analysis workflow. Each section runs the SQL against the loaded MySQL table, displays the result, and generates a concise finding from the returned rows.

**Database required:** MySQL 8.0+, database `sql_project_p2`, table `retail_sales` loaded from the project dataset. The notebook does not fabricate sample totals. The finding cells are driven by query results from your database.

**Before running:** set environment variables `MYSQL_HOST`, `MYSQL_PORT`, `MYSQL_USER`, `MYSQL_PASSWORD`, `MYSQL_DATABASE` (default database: `sql_project_p2`).

## 1. Connect and validate the source table

The connection uses environment variables so credentials are not saved in this notebook. The validation cell checks row count, date coverage, transaction ID uniqueness and basic sales totals before interpreting any query.

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

required = ["MYSQL_HOST", "MYSQL_USER", "MYSQL_PASSWORD"]
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise EnvironmentError(
        "Set these environment variables before running: " + ", ".join(missing)
    )

port = os.getenv("MYSQL_PORT", "3306")
database = os.getenv("MYSQL_DATABASE", "sql_project_p2")
connection_url = (
    f"mysql+pymysql://{os.environ['MYSQL_USER']}:{os.environ['MYSQL_PASSWORD']}"
    f"@{os.environ['MYSQL_HOST']}:{port}/{database}"
)
engine = create_engine(connection_url, pool_pre_ping=True)

def run_query(sql):
    """Run a read-only analysis query and return its result as a DataFrame."""
    return pd.read_sql_query(text(sql), engine)

source_check = run_query("""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT transaction_id) AS distinct_transaction_ids,
    MIN(sale_date) AS first_sale_date,
    MAX(sale_date) AS last_sale_date,
    SUM(total_sale) AS total_sales,
    SUM(quantity) AS total_units
FROM retail_sales
""")
display(source_check)

if source_check.empty or int(source_check.loc[0, "row_count"]) == 0:
    raise ValueError("retail_sales has no rows; load the source dataset first.")
if source_check.loc[0, "row_count"] != source_check.loc[0, "distinct_transaction_ids"]:
    print("DATA QUALITY FLAG: transaction_id is not unique; verify the source grain before treating COUNT(*) as transaction count.")
else:
    print("DATA QUALITY CHECK: transaction_id is unique across the loaded rows.")

## 2. Q1 — Transactions on 5 November 2022

**SQL concept:** exact date filtering. **Business use:** audit activity on a specific date.

In [ ]:
q1 = run_query("""
SELECT transaction_id, sale_date, sale_time, customer_id, category, quantity, total_sale
FROM retail_sales
WHERE sale_date = '2022-11-05'
ORDER BY sale_time
""")
display(q1)
print(f"Finding: {len(q1):,} transaction rows occurred on 5 November 2022; their combined sales value was {q1['total_sale'].sum():,.2f}.")

## 3. Q2 — High-quantity Clothing transactions in November 2022

**SQL concept:** half-open date range, multiple predicates. **Business use:** investigate bulk purchases and unusual quantity patterns.

In [ ]:
q2 = run_query("""
SELECT transaction_id, sale_date, sale_time, customer_id, quantity, total_sale
FROM retail_sales
WHERE category = 'Clothing'
  AND sale_date >= '2022-11-01'
  AND sale_date < '2022-12-01'
  AND quantity > 10
ORDER BY sale_date, sale_time
""")
display(q2)
print(f"Finding: {len(q2):,} Clothing transactions met the quantity > 10 rule in November 2022; combined sales value: {q2['total_sale'].sum():,.2f}. Review individual rows before labelling them errors or bulk orders.")

## 4. Q3 — Sales and average transaction value by category

**SQL concept:** aggregation and group-level comparison. **Business use:** compare category scale and basket value.

In [ ]:
q3 = run_query("""
SELECT category, SUM(total_sale) AS net_sales,
       COUNT(*) AS transaction_count,
       ROUND(AVG(total_sale), 2) AS average_transaction_value
FROM retail_sales
GROUP BY category
ORDER BY net_sales DESC
""")
display(q3)
if not q3.empty:
    r = q3.iloc[0]
    print(f"Finding: {r['category']} leads recorded category sales at {r['net_sales']:,.2f} across {int(r['transaction_count']):,} rows. Compare average transaction value as well before making assortment decisions.")

## 5. Q4 — Average age of customers buying Beauty

**SQL concept:** filtered aggregate. **Business use:** descriptive profile of buyers; this is not evidence that age causes category preference.

In [ ]:
q4 = run_query("""
SELECT ROUND(AVG(age), 2) AS average_customer_age,
       COUNT(*) AS transaction_rows,
       COUNT(age) AS rows_with_age
FROM retail_sales
WHERE category = 'Beauty'
""")
display(q4)
print("Finding: the displayed average describes Beauty transaction rows with a non-null age; it is not necessarily the average age of unique Beauty customers.")

## 6. Q5 — Transactions above 1,000 sales value

**SQL concept:** threshold filtering. **Business use:** review high-value transactions for service, basket or audit follow-up.

In [ ]:
q5 = run_query("""
SELECT transaction_id, sale_date, customer_id, category, quantity, total_sale
FROM retail_sales
WHERE total_sale > 1000
ORDER BY total_sale DESC
""")
display(q5)
print(f"Finding: {len(q5):,} transactions exceed 1,000; combined sales value: {q5['total_sale'].sum():,.2f}. High value alone does not imply an error.")

## 7. Q6 — Sales, units and transaction rows by gender and category

**SQL concept:** multi-column GROUP BY. **Business use:** compare observed category mix without inferring causation.

In [ ]:
q6 = run_query("""
SELECT category, gender, COUNT(*) AS transaction_count,
       SUM(quantity) AS units_sold, SUM(total_sale) AS total_sales
FROM retail_sales
GROUP BY category, gender
ORDER BY category, total_sales DESC
""")
display(q6)
print("Finding: compare total_sales, units_sold and transaction_count together; a larger sales total may reflect more transactions, more units, or higher basket value.")

## 8. Q7 — Highest-sales month within each year

**SQL concept:** CTE + DENSE_RANK. Ties are retained. **Business use:** identify strong sales periods for seasonal planning.

In [ ]:
q7 = run_query("""
WITH monthly_sales AS (
    SELECT YEAR(sale_date) AS sale_year, MONTH(sale_date) AS sale_month,
           SUM(total_sale) AS total_sales,
           AVG(total_sale) AS average_transaction_value,
           COUNT(*) AS transaction_count
    FROM retail_sales
    GROUP BY YEAR(sale_date), MONTH(sale_date)
),
ranked_months AS (
    SELECT *, DENSE_RANK() OVER (
        PARTITION BY sale_year ORDER BY total_sales DESC
    ) AS sales_rank
    FROM monthly_sales
)
SELECT sale_year, sale_month, total_sales,
       ROUND(average_transaction_value, 2) AS average_transaction_value,
       transaction_count
FROM ranked_months
WHERE sales_rank = 1
ORDER BY sale_year, sale_month
""")
display(q7)
print("Finding: each returned month is tied for the highest recorded monthly sales in its year. Use transaction count and average transaction value to understand whether the peak reflects volume or basket value.")

## 9. Q8 — Top five customers by spend

**SQL concept:** aggregation + ORDER BY + LIMIT. **Business use:** customer concentration and follow-up; protect customer identifiers in shared reports.

In [ ]:
q8 = run_query("""
SELECT customer_id, SUM(total_sale) AS total_sales,
       COUNT(*) AS transaction_count, SUM(quantity) AS units_purchased
FROM retail_sales
WHERE customer_id IS NOT NULL
GROUP BY customer_id
ORDER BY total_sales DESC
LIMIT 5
""")
display(q8)
if not q8.empty:
    total = float(source_check.loc[0, "total_sales"] or 0)
    top5 = float(q8["total_sales"].sum())
    share = (100 * top5 / total) if total else None
    print(f"Finding: the top five customers account for {share:.2f}% of recorded sales." if share is not None else "Finding: overall sales is zero; customer share cannot be calculated.")

## 10. Q9 — Distinct customers by category

**SQL concept:** COUNT(DISTINCT). **Business use:** compare category reach rather than only sales value.

In [ ]:
q9 = run_query("""
SELECT category, COUNT(DISTINCT customer_id) AS unique_customers
FROM retail_sales
GROUP BY category
ORDER BY unique_customers DESC
""")
display(q9)
if not q9.empty:
    r = q9.iloc[0]
    print(f"Finding: {r['category']} has the largest number of distinct recorded customers ({int(r['unique_customers']):,}). This is category reach, not customer loyalty.")

## 11. Q10 — Sales and transaction rows by time-of-day shift

**Shift definition:** Morning before 12:00; Afternoon from 12:00 to before 18:00; Evening from 18:00 onward. **Business use:** staffing and opening-hour review.

In [ ]:
q10 = run_query("""
WITH sales_by_shift AS (
    SELECT *,
        CASE WHEN sale_time < '12:00:00' THEN 'Morning'
             WHEN sale_time < '18:00:00' THEN 'Afternoon'
             ELSE 'Evening' END AS shift
    FROM retail_sales
)
SELECT shift, COUNT(*) AS transaction_count,
       SUM(total_sale) AS total_sales,
       ROUND(AVG(total_sale), 2) AS average_transaction_value
FROM sales_by_shift
GROUP BY shift
ORDER BY CASE shift WHEN 'Morning' THEN 1 WHEN 'Afternoon' THEN 2 ELSE 3 END
""")
display(q10)
if not q10.empty:
    r = q10.loc[q10["total_sales"].idxmax()]
    print(f"Finding: {r['shift']} has the highest recorded sales ({r['total_sales']:,.2f}). Validate operating hours and staffing costs before changing schedules.")

## 12. Q11 — Category share of total revenue

**SQL concept:** ratio to overall total with NULLIF. **Business use:** category prioritisation and revenue mix.

In [ ]:
q11 = run_query("""
SELECT category, SUM(total_sale) AS category_sales,
       ROUND(100 * SUM(total_sale) /
       NULLIF((SELECT SUM(total_sale) FROM retail_sales), 0), 2)
       AS revenue_share_percentage
FROM retail_sales
GROUP BY category
ORDER BY revenue_share_percentage DESC
""")
display(q11)
if not q11.empty:
    r = q11.iloc[0]
    print(f"Finding: {r['category']} contributes the largest category revenue share ({r['revenue_share_percentage']}%). Revenue share alone does not indicate profit margin.")

## 13. Q12 — Weekday sales pattern

**SQL concept:** date functions and grouping. **Business use:** compare activity by weekday; weekday numbering is MySQL-specific.

In [ ]:
q12 = run_query("""
SELECT DAYOFWEEK(sale_date) AS weekday_number,
       DAYNAME(sale_date) AS weekday_name,
       COUNT(*) AS transaction_count,
       SUM(total_sale) AS total_sales,
       ROUND(AVG(total_sale), 2) AS average_transaction_value
FROM retail_sales
GROUP BY DAYOFWEEK(sale_date), DAYNAME(sale_date)
ORDER BY weekday_number
""")
display(q12)
if not q12.empty:
    r = q12.loc[q12["total_sales"].idxmax()]
    print(f"Finding: {r['weekday_name']} has the highest recorded weekday sales ({r['total_sales']:,.2f}). Check whether each weekday has comparable coverage before comparing totals.")

## 14. Q13 — Revenue concentration across customer spend quintiles

**SQL concept:** NTILE(5) over customer totals. Quintile 1 is the highest-spending group because the ordering is descending. **Business use:** quantify dependence on high-spend customers.

In [ ]:
q13 = run_query("""
WITH customer_sales AS (
    SELECT customer_id, SUM(total_sale) AS total_sales
    FROM retail_sales
    WHERE customer_id IS NOT NULL
    GROUP BY customer_id
),
ranked_customers AS (
    SELECT *, NTILE(5) OVER (ORDER BY total_sales DESC) AS spend_quintile
    FROM customer_sales
)
SELECT spend_quintile, COUNT(*) AS customer_count,
       SUM(total_sales) AS segment_sales,
       ROUND(100 * SUM(total_sales) /
       NULLIF((SELECT SUM(total_sales) FROM customer_sales), 0), 2)
       AS sales_share_percentage
FROM ranked_customers
GROUP BY spend_quintile
ORDER BY spend_quintile
""")
display(q13)
if not q13.empty:
    top = q13.loc[q13["spend_quintile"] == 1]
    if not top.empty:
        r = top.iloc[0]
        print(f"Finding: the highest-spending customer quintile contains {int(r['customer_count']):,} customers and contributes {r['sales_share_percentage']}% of customer-linked sales. A high share suggests concentration risk worth monitoring.")

## 15. Executive summary — automatic findings

This cell assembles the strongest results from the query outputs. It is generated from the connected dataset rather than fixed example numbers.

In [ ]:
summary = []
if not q3.empty:
    r = q3.iloc[0]
    summary.append(f"Top sales category: {r['category']} ({r['net_sales']:,.2f}).")
if not q7.empty:
    summary.append("Peak sales month(s) by year: " + "; ".join(
        f"{int(r.sale_year)}-{int(r.sale_month):02d} ({r.total_sales:,.2f})"
        for r in q7.itertuples(index=False)
    ) + ".")
if not q10.empty:
    r = q10.loc[q10["total_sales"].idxmax()]
    summary.append(f"Highest-sales shift: {r['shift']} ({r['total_sales']:,.2f}).")
if not q11.empty:
    r = q11.iloc[0]
    summary.append(f"Largest revenue-share category: {r['category']} ({r['revenue_share_percentage']}%).")
if not q13.empty:
    top = q13.loc[q13["spend_quintile"] == 1]
    if not top.empty:
        r = top.iloc[0]
        summary.append(f"Top-spend quintile share: {r['sales_share_percentage']}% of customer-linked sales.")
print("\n".join(f"- {item}" for item in summary))
print("\nBusiness action: use these findings to prioritise category review, staffing analysis and customer-concentration monitoring. Validate the reporting period, data quality and profitability before changing budgets or operations.")

## Interpretation checklist

- Confirm the source dataset, reporting period and row grain.
- Distinguish transaction rows, unique customers, units and sales value.
- Treat rankings and correlations as descriptive; they do not establish cause.
- Compare sales with gross margin, returns, discounts and stock availability before making commercial recommendations.
- Refresh the notebook after the source table changes so the displayed findings remain reproducible.